# fMRI data matching and preparation
Maria B. Jelen

This script:
1) Loads in the fMRI matrices from stored files, cleans them from NaNs, formats naming
2) Performs thresholding using top 20% connections per row (Margulies et al. 2016)
3) Saves files for PLS analyses

In [2]:
import pandas as pd
import os
import numpy as np

In [ ]:
# Specify path to fMRI matrices and SOM island membership file
scan_folder_path = r'\\cbsu\data\Imaging\projects\external\abcd\data\BIDS_Maria\baselineYear1Arm1\yeo100'
membership_df = pd.read_csv('som_island_membership.csv')

# fMRI file import and cleaning
all_filenames = os.listdir(scan_folder_path)
scan_ids = set([f.split('sub-')[1].split('_')[0] for f in all_filenames if f.startswith('sub-')])

# Check how many participants have available scans
membership_df['src_subject_id'] = membership_df['src_subject_id'].str.replace('_', '') # sync ID formatting
membership_df_with_scans = membership_df[membership_df['src_subject_id'].isin(scan_ids)].copy()
print(f"Found {len(membership_df_with_scans)} participants shared with behavioural data with available scans. Expected: 4326.")

# Constructing matrices for those participants
ids_to_process = membership_df_with_scans['src_subject_id'].tolist()
print(f"\nBuilding fMRI matrix for {len(ids_to_process)} participants")

all_participant_edges = []
processed_ids = []
for participant_id in ids_to_process:
    filename = f"sub-{participant_id}_ses-baselineYear1Arm1_task-rest_space-yeo100_FC.csv"
    file_path = os.path.join(scan_folder_path, filename)
    if os.path.exists(file_path):
        conn_matrix = pd.read_csv(file_path, header=None).values
        indices = np.triu_indices(n=100, k=1)
        edges = conn_matrix[indices]
        all_participant_edges.append(edges)
        processed_ids.append(participant_id)

edge_column_names = [f'edge_{i+1}' for i in range(4950)]
fmri_df = pd.DataFrame(all_participant_edges, columns=edge_column_names)
fmri_df.insert(0, 'src_subject_id', processed_ids)

Found 4326 participants shared with behavioural data with available scans. Expected: 4326.

Building fMRI matrix for 4326 participants


In [3]:
# Cleaning data

# Replace any infinity values with NaN
fmri_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Find participants with any NaN
ids_with_nan = fmri_df.loc[fmri_df.isnull().any(axis=1), 'src_subject_id']
print(f"Found {len(ids_with_nan)}  participants with NaN or Inf in scan data.")

# Exclude these participants from BOTH profile membership and scan dataframes
fmri_df_complete = fmri_df[~fmri_df['src_subject_id'].isin(ids_with_nan)]
membership_df_complete = membership_df_with_scans[~membership_df_with_scans['src_subject_id'].isin(ids_with_nan)]

ids_with_nan_afterfclean = fmri_df_complete.loc[fmri_df_complete.isnull().any(axis=1), 'src_subject_id']
print(f"After cleaning, found {len(ids_with_nan_afterfclean)} participants with NaN in scan data.")

# Verify dataframe shape
print(f"Final fmri_df_complete shape: {fmri_df_complete.shape}")
print(f"Final membership_df_complete shape: {membership_df_complete.shape}")

# a. Save the cleaned fMRI data
fmri_df_complete.to_csv('fmri_df_cleaned.csv', index=False)
print(f"Saved 'fmri_df_cleaned.csv' with shape {fmri_df_complete.shape}")

# b. Save the membership data
membership_df_complete.to_csv('membership_df_cleaned.csv', index=False)
print(f"Saved 'membership_df_cleaned.csv' with shape {membership_df_complete.shape}")

Found 62  participants with NaN or Inf in scan data.
After cleaning, found 0 participants with NaN in scan data.
Final fmri_df_complete shape: (4264, 4951)
Final membership_df_complete shape: (4264, 11)
Saved 'fmri_df_cleaned.csv' with shape (4264, 4951)
Saved 'membership_df_cleaned.csv' with shape (4264, 11)


In [4]:
# Load data and align
membership_df = pd.read_csv('membership_df_cleaned.csv')
fmri_df = pd.read_csv('fmri_df_cleaned.csv')
analysis_df = pd.merge(membership_df, fmri_df, on='src_subject_id', how='inner')
print(f" {len(analysis_df)} participants present in both files")

 4264 participants present in both files


In [ ]:
# Cleaning membership matrix which will become Y matrix for PLS analysis

from collections import defaultdict

# Pool together individual profile zones into joint profile subtypes (e.g. 'somaticising 1' and 'somaticising 2' become 'somaticising' column)
cols_to_pool = defaultdict(list)
island_cols = [col for col in membership_df.columns if col != 'src_subject_id']
for col in island_cols:
    base_name = '_'.join(col.split('_')[:-1])
    cols_to_pool[base_name].append(col)

# Create a binary matrix where a subject is marked 1 if they belong to any cluster within that profile
pooled_Y_df = pd.DataFrame()
pooled_Y_df['src_subject_id'] = analysis_df['src_subject_id']
for base_name, original_cols in cols_to_pool.items():
    pooled_Y_df[base_name] = analysis_df[original_cols].any(axis=1).astype(int)

# Identify if any column is (nearly) empty - none detected, so none are removed
Y_data_to_clean = pooled_Y_df.drop('src_subject_id', axis=1)
low_count_cols = Y_data_to_clean.columns[Y_data_to_clean.sum() < 3]
behavioral_df_pooled = Y_data_to_clean.drop(columns=low_count_cols)
print(f"Final membership (Y) matrix created with {behavioral_df_pooled.shape[1]} features.")

# Saving file with participant ID
behavioral_df_pooled.insert(0, 'src_subject_id', analysis_df['src_subject_id'])

#behavioral_df_pooled.to_csv('behavioral_matrix_pooled.csv', index=False)
print(f"Saved 'behavioral_matrix_pooled.csv' with shape {behavioral_df_pooled.shape}")


Final membership (Y) matrix created with 7 features.
Saved 'behavioral_matrix_pooled.csv' with shape (4264, 8)


In [6]:
# Create neuroimaging matrix using top 20% connections per row - this will become X matrix in rsFC PLS
X_df_unthresholded = analysis_df.filter(like='edge_')
edge_data_unthresholded = X_df_unthresholded.values
hybrid_edges_list = []
n_nodes = 100
indices = np.triu_indices(n=n_nodes, k=1)

for i in range(len(edge_data_unthresholded)):
    conn_matrix = np.zeros((n_nodes, n_nodes))
    conn_matrix[indices] = edge_data_unthresholded[i]
    conn_matrix = conn_matrix + conn_matrix.T
    
    row_wise_mask = np.zeros_like(conn_matrix, dtype=bool)
    for j in range(n_nodes):
        row_data = conn_matrix[j, :]
        threshold_value = np.percentile(row_data, 80)
        row_wise_mask[j, row_data >= threshold_value] = True
        
    symmetric_mask = row_wise_mask | row_wise_mask.T
    hybrid_thresholded_matrix = conn_matrix * symmetric_mask
    hybrid_edges_list.append(hybrid_thresholded_matrix[indices])

X_df_hybrid = pd.DataFrame(hybrid_edges_list, columns=X_df_unthresholded.columns)

# Cleaning from any remaining Inf or NaN values
fmri_data_array = X_df_hybrid.values

# Check for non-finite values (NaN or inf) within columns
has_non_finite = ~np.isfinite(fmri_data_array).all(axis=0)

# Check for near-zero variance columns
stds = np.nanstd(fmri_data_array, axis=0, ddof=1)
has_low_variance = stds < 1e-6

# Combine checks: point out column if either criterion is true
is_problematic = has_non_finite | has_low_variance

# If any columns exist, get names to flag them
cols_to_drop = X_df_hybrid.columns[is_problematic]

if len(cols_to_drop) > 0:
    print(f"Found and removed {len(cols_to_drop)} problem columns.")
    fmri_df_hybrid = X_df_hybrid.drop(columns=cols_to_drop)
else:
    print("No problem columns found.")
    fmri_df_hybrid = X_df_hybrid

print(f"Final fMRI matrix created with {fmri_df_hybrid.shape[1]} features.")

# saving with ID
fmri_df_hybrid.insert(0, 'src_subject_id', analysis_df['src_subject_id'])

# fmri_df_hybrid.to_csv('fmri_threshold_80.csv', index=False)
print(f"Saved 'fmri_threshold_80.csv' with shape {fmri_df_hybrid.shape}")

No problem columns found.
Final fMRI matrix created with 4950 features.
Saved 'fmri_threshold_80.csv' with shape (4264, 4951)
